# PS5 — Hardware-OOS Failure Spine + Per-Serial Component Table (notebook-only, v2)

**What this does.** Rebuilds the PS5 failure event as **any hardware OOS 'Set' that lasted ≥ `MIN_OUTAGE_MIN` minutes** — uniformly for TVM / gates / validators — read *directly* from `silver.device_event_enriched` (the 190.9M-row OOS stream), instead of the chargeable-bounded `silver.device_failures` spine. It also builds a per-serial component table with **true per-component OOS counts** and captures **chargeable** events as an analysis column (device + serial level).

**v2 change — the min-outage filter.** Raw hardware OOS counts every transient comms blip (sub-minute self-recovering faults), which made validators look like they "fail" daily → a degenerate ~1-day RUL. This version computes each OOS event's **outage duration from its Set→Clear lifecycle** and drops anything shorter than `MIN_OUTAGE_MIN` (default 5 min), so the survival model sees real outages, not blips.

**Guardrails (read them):**
- **Read-only from silver.** Never writes to any `mars_dev.silver.*` / `mars_dev.gold.*` table. Outputs go to a local folder and are wired into the engine **in memory**.
- **Event = hardware OOS Set (≥ MIN_OUTAGE_MIN)**, NOT chargeable. Chargeable (`FAILURE_LEVEL>0 AND EXCLUDED=0` from `kpi_avail_enriched`) is captured as `total_chargeable_events` / `chargeable_pct` for analysis only.
- **Validators (BMV)** have no chargeable stream → `total_chargeable_events` = 0 by construction.
- **Per-serial is counts + age, not a survival curve** (component-level survival is 100% censored per the catalog).
- **Schema-adaptive + self-reporting.** Column names are resolved at runtime and printed; the min-outage diagnostic shows episodes dropped per type so you can verify the effect.

**Run order:** cells top-to-bottom in the **same kernel as your PS5 engine cell** (so `CONFIG` exists for the wiring cell). Then re-run `main(synthetic=False)`.


In [ ]:
# ── Cell 1 — Spark bootstrap (reuse an active session, else local S3A) ─────────
# Safest: run this in the SAME kernel as your working PS1 validator notebook (its Spark
# session + Java are already configured) — this cell will just reuse it.
import os, sys, subprocess, importlib

def _pip(*pkgs):
    try: ip = get_ipython()
    except NameError: ip = None
    if ip is not None: ip.run_line_magic("pip", "install -q " + " ".join(pkgs))
    else: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    importlib.invalidate_caches()

try:
    import pyspark  # noqa
except ImportError:
    _pip("pyspark==3.5.9")

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"   # matches Spark 3.5 / Hadoop 3.3.x

spark = SparkSession.getActiveSession()
if spark is None:
    os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 22g pyspark-shell")
    spark = (SparkSession.builder.appName("PS5_OOS_Spine_Builder")
             .master("local[*]")
             .config("spark.driver.memory", "22g")
             .config("spark.driver.maxResultSize", "6g")
             .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
             .config("spark.sql.adaptive.enabled", "true")
             .config("spark.jars.packages", HADOOP_AWS_PACKAGE)
             .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                     "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
             .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
             .getOrCreate())
print("Spark:", spark.version, "| master:", spark.sparkContext.master)


In [ ]:
# ── Cell 2 — Config (paths, window, scope, min-outage). Mirrors the PS5 engine. ─
GOLD_BUCKET     = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
SCHEME          = "s3a"   # flip to "s3" if your Spark reads use that scheme
def SILVER(t):  return f"{SCHEME}://{GOLD_BUCKET}/chicago/silver/{t}"

RUN_DATE        = "2026-04-11"     # ODS watermark; drop events after this (matches engine RUN_DATE)
TELEMETRY_START = "2024-01-01"     # device_event_enriched window start
SCOPE           = ["TVM", "GATE", "VALIDATOR"]

MIN_OUTAGE_MIN  = 5.0              # drop OOS shorter than this many minutes (transient blips). 0 = keep all.

# operational (non-hardware) events to exclude, per the Data Foundation doc
OPS_EXCLUDE = ["emp logon", "in service", "maintenance mode", "vehicle ignition off", "cmdoosbytables"]

OUT_LOCAL = "PS5_oos_spine_outputs"   # LOCAL scratch (NOT a silver/gold table)
os.makedirs(OUT_LOCAL, exist_ok=True)
print("scope:", SCOPE, "| window:", TELEMETRY_START, "->", RUN_DATE, "| min_outage_min:", MIN_OUTAGE_MIN)


In [ ]:
# ── Cell 3 — Helpers + schema probe (verify the adaptive column matches) ─────
def resolve(available, wanted):
    """Case-insensitive: return {role: actual_col_or_None} from candidate lists."""
    low = {c.lower(): c for c in available}
    return {role: next((low[c.lower()] for c in cands if c.lower() in low), None)
            for role, cands in wanted.items()}

def truthy(colname):
    c = F.col(colname)
    return (c.cast("boolean") == F.lit(True)) | \
           (F.lower(F.trim(c.cast("string"))).isin("true", "1", "t", "y", "yes"))

def cols_of(table):
    try:
        return spark.read.parquet(SILVER(table)).limit(1).columns
    except Exception as e:
        print(f"  [probe] {table}: {type(e).__name__} {e}"); return []

for t in ["device_event_enriched", "kpi_avail_enriched", "hw_config_current",
          "dim_failure_level", "maintenance_ledger", "device_mttr"]:
    c = cols_of(t)
    print(f"\n### {t} ({len(c)} cols)\n" + ", ".join(c))


In [ ]:
# ── Cell 4 — Hardware-OOS-Set spine + MIN-OUTAGE filter (drop transient blips) ─
# Order matters: window+scope first (keep Set AND Clear so we can pair for duration),
# derive outage_min (duration col -> clear-ts col -> Set/Clear lead-pairing), THEN apply
# the OOS/device-fault/commanded/maintenance/ops filters on the Set rows, THEN min-outage.
ev = spark.read.parquet(SILVER("device_event_enriched"))
Rev = resolve(ev.columns, {
    "dev":   ["DEVICE_ID", "device_id"],
    "key":   ["DEVICE_KEY", "device_key"],
    "cat":   ["mars_device_category", "device_category", "DEVICE_CATEGORY"],
    "dtm":   ["EVENT_DTM", "event_dtm", "EVENT_START_DTM", "FAULT_SET_DTM", "START_DTM", "EVENT_TS"],
    "state": ["FAULT_STATE", "fault_state", "AE_FAULT_STATE", "EVENT_FAULT_STATE"],
    "oos":   ["is_counted_oos", "counted_as_oos", "is_oos_event", "is_oos", "is_counted_as_oos", "counted_oos"],
    "devfault": ["is_device_fault", "IS_DEVICE_FAULT"],
    "cmd":   ["is_commanded_oos", "IS_COMMANDED_OOS"],
    "maint": ["is_maintenance_oos", "IS_MAINTENANCE_OOS"],
    "etname":["EVENT_TYPE_NAME", "event_type_name", "EVENT_NAME", "event_description", "EVENT_DESC"],
    "ser":   ["COMPONENT_SERIAL_NBR", "component_serial_nbr"],
    "clear": ["CLEAR_DTM", "clear_dtm", "EVENT_CLEAR_DTM", "FAULT_CLEAR_DTM", "RESOLVED_DTM", "END_DTM", "end_dtm"],
    "dur":   ["outage_duration_min", "duration_min", "outage_min", "mttr_min", "mttr_minutes",
              "duration_minutes", "outage_minutes"],
})
print("resolved device_event_enriched ->", {k: v for k, v in Rev.items()})
assert Rev["dev"] and Rev["dtm"] and Rev["cat"], "DEVICE_ID / event date / category not found — check the probe above"

ecol = F.col(Rev["dtm"]).cast("timestamp")
hw = (ev.withColumn("_d", F.to_date(F.col(Rev["dtm"])))
        .filter((F.col("_d") >= F.lit(TELEMETRY_START)) & (F.col("_d") <= F.lit(RUN_DATE)))
        .filter(F.upper(F.col(Rev["cat"]).cast("string")).isin(*SCOPE)))

# ---- derive outage_min (minutes) per event, adaptively ----
if Rev["dur"]:
    hw = hw.withColumn("outage_min", F.col(Rev["dur"]).cast("double")); DUR_SRC = f"column {Rev['dur']}"
elif Rev["clear"]:
    hw = hw.withColumn("outage_min",
                       (F.col(Rev["clear"]).cast("timestamp").cast("long") - ecol.cast("long")) / 60.0)
    DUR_SRC = f"{Rev['clear']} - {Rev['dtm']}"
elif Rev["state"]:
    # pair each Set with its next event per device(+component); if that next event is a Clear, gap = outage
    part = [F.col(Rev["dev"]).cast("string").alias("_pdev")]
    keyc = ["_pdev"] + (["_pser"] if Rev["ser"] else [])
    tmp = (hw.withColumn("_pdev", F.col(Rev["dev"]).cast("string"))
             .withColumn("_pser", (F.col(Rev["ser"]).cast("string") if Rev["ser"] else F.lit("NA")))
             .withColumn("_st", F.lower(F.trim(F.col(Rev["state"]).cast("string"))))
             .filter(F.col("_st").isin("set", "s", "clear", "c")))   # lifecycle rows only -> cheaper sort
    w = Window.partitionBy(*keyc).orderBy(ecol)
    tmp = (tmp.withColumn("_next_ts", F.lead(ecol).over(w))
              .withColumn("_next_st", F.lead("_st").over(w))
              .withColumn("outage_min",
                          F.when(F.col("_st").isin("set", "s") & F.col("_next_st").isin("clear", "c"),
                                 (F.col("_next_ts").cast("long") - ecol.cast("long")) / 60.0)))
    hw = tmp; DUR_SRC = "Set->Clear pairing (lead over device+component)"
else:
    hw = hw.withColumn("outage_min", F.lit(None).cast("double")); DUR_SRC = "UNAVAILABLE"
print("outage source:", DUR_SRC)

# ---- keep the fault ONSET (Set) rows, apply the OOS/device-fault/exclusion filters ----
onset = hw
if Rev["state"]:
    onset = onset.filter(F.lower(F.trim(F.col(Rev["state"]).cast("string"))).isin("set", "s"))
applied = ["window", "scope"]
if Rev["oos"]:      onset = onset.filter(truthy(Rev["oos"]));   applied.append(f"{Rev['oos']}=T")
else:               print("  [warn] no counted-as-OOS flag; relying on state+device_fault+ops-exclude")
if Rev["devfault"]: onset = onset.filter(truthy(Rev["devfault"])); applied.append(f"{Rev['devfault']}=T")
if Rev["cmd"]:      onset = onset.filter(~truthy(Rev["cmd"]));   applied.append(f"NOT {Rev['cmd']}")
if Rev["maint"]:    onset = onset.filter(~truthy(Rev["maint"])); applied.append(f"NOT {Rev['maint']}")
if Rev["etname"]:
    for nm in OPS_EXCLUDE:
        onset = onset.filter(~F.lower(F.col(Rev["etname"]).cast("string")).contains(nm))
    applied.append("exclude ops")
print("filters applied:", applied)

base_all = onset.select(
    F.col(Rev["dev"]).cast("string").alias("DEVICE_ID"),
    (F.col(Rev["key"]).cast("string") if Rev["key"] else F.col(Rev["dev"]).cast("string")).alias("DEVICE_KEY"),
    F.upper(F.col(Rev["cat"]).cast("string")).alias("mars_device_category"),
    F.col("_d").alias("failure_date"),
    (F.col(Rev["ser"]).cast("string") if Rev["ser"] else F.lit(None).cast("string")).alias("COMPONENT_SERIAL_NBR"),
    F.col("outage_min").cast("double").alias("outage_min"),
).persist()

# ---- MIN-OUTAGE filter: keep outage>=MIN_OUTAGE_MIN OR null (null=unpaired/open=real long outage) ----
if MIN_OUTAGE_MIN and MIN_OUTAGE_MIN > 0 and DUR_SRC != "UNAVAILABLE":
    base = base_all.filter((F.col("outage_min") >= F.lit(MIN_OUTAGE_MIN)) | F.col("outage_min").isNull())
else:
    if MIN_OUTAGE_MIN and DUR_SRC == "UNAVAILABLE":
        print("  [warn] no duration/clear/state column -> min-outage NOT applied. "
              "Tell me the real clear-time or duration column (or device_mttr keys) and I'll wire it.")
    base = base_all
base = base.persist()

# ---- diagnostic: episodes dropped by the min-outage filter, per type ----
def _epi(d): return d.dropDuplicates(["DEVICE_ID", "failure_date"])
b = _epi(base_all).groupBy("mars_device_category").agg(F.count("*").alias("episodes_raw"))
a = _epi(base).groupBy("mars_device_category").agg(F.count("*").alias("episodes_kept"))
b.join(a, "mars_device_category", "outer").orderBy("mars_device_category").show()
print(f"min-outage >= {MIN_OUTAGE_MIN} min applied via [{DUR_SRC}]; nulls (open/never-cleared) kept as real outages.")

# ---- aggregations used downstream ----
epi_dev  = _epi(base).persist()
oos_dev  = epi_dev.groupBy("DEVICE_ID", "DEVICE_KEY", "mars_device_category") \
                  .agg(F.count("*").alias("total_hardware_oos_events"))
comp_oos = (base.filter(F.col("COMPONENT_SERIAL_NBR").isNotNull())
                .dropDuplicates(["DEVICE_ID", "COMPONENT_SERIAL_NBR", "failure_date"])
                .groupBy("DEVICE_ID", "COMPONENT_SERIAL_NBR")
                .agg(F.count("*").alias("component_oos_events")))
print("kept episodes:", epi_dev.count(), "| devices with OOS:", oos_dev.count())


In [ ]:
# ── Cell 5 — Chargeable device counts from kpi_avail_enriched (analysis only) ─
av = spark.read.parquet(SILVER("kpi_avail_enriched"))
Rav = resolve(av.columns, {
    "dev":  ["DEVICE_ID", "device_id"],
    "fl":   ["FAILURE_LEVEL", "failure_level"],
    "exc":  ["EXCLUDED", "excluded"],
    "state":["FAULT_STATE", "fault_state"],
    "dtm":  ["START_DTM", "start_dtm", "transit_day", "TRANSIT_DAY"],
})
print("resolved kpi_avail_enriched ->", {k: v for k, v in Rav.items()})
ch = av
if Rav["fl"]:    ch = ch.filter(F.col(Rav["fl"]).cast("double") > 0)                                   # chargeable = FAILURE_LEVEL>0
if Rav["exc"]:   ch = ch.filter((F.col(Rav["exc"]).cast("double") == 0) | F.col(Rav["exc"]).isNull())  # AND not EXCLUDED
if Rav["state"]: ch = ch.filter(F.lower(F.trim(F.col(Rav["state"]).cast("string"))).isin("set", "s"))
if Rav["dtm"]:   ch = ch.withColumn("_cd", F.to_date(F.col(Rav["dtm"]))).filter(F.col("_cd") <= F.lit(RUN_DATE))
else:            ch = ch.withColumn("_cd", F.lit(None))
charge_dev = (ch.dropDuplicates([Rav["dev"], "_cd"])
                .groupBy(F.col(Rav["dev"]).cast("string").alias("DEVICE_ID"))
                .agg(F.count("*").alias("total_chargeable_events")))
print("devices with chargeable:", charge_dev.count())


In [ ]:
# ── Cell 6 — Device summary: OOS + chargeable + pct + median outage ──────────
outage_dev = base.groupBy("DEVICE_ID").agg(
    F.round(F.expr("percentile_approx(outage_min, 0.5)"), 1).alias("median_outage_min"))
summary = (oos_dev.join(charge_dev, "DEVICE_ID", "left")
                  .join(outage_dev, "DEVICE_ID", "left")
                  .fillna({"total_chargeable_events": 0})
                  .withColumn("chargeable_pct",
                              F.round(100.0 * F.col("total_chargeable_events") /
                                      F.greatest(F.col("total_hardware_oos_events"), F.lit(1)), 2))
                  .persist())
print("device summary rows:", summary.count())
summary.groupBy("mars_device_category").agg(
    F.countDistinct("DEVICE_ID").alias("devices"),
    F.sum("total_hardware_oos_events").alias("oos_events"),
    F.sum("total_chargeable_events").alias("chargeable_events"),
    F.round(100.0*F.sum("total_chargeable_events")/F.greatest(F.sum("total_hardware_oos_events"), F.lit(1)), 2).alias("chargeable_pct"),
    F.round(F.avg("median_outage_min"), 1).alias("avg_dev_median_outage_min")
).show()
print("Sanity: TVM/gate chargeable_pct ~6-8% (per serial_id_device_id.xlsx); VALIDATOR = 0. "
      "median outage should now be >= a few minutes (blips filtered).")


In [ ]:
# ── Cell 7 — Per-serial component table (hw_config x device summary + age) ────
hw_cfg = spark.read.parquet(SILVER("hw_config_current"))
Rhw = resolve(hw_cfg.columns, {
    "dev":  ["DEVICE_ID", "device_id"],
    "cat":  ["mars_device_category", "device_category"],
    "ser":  ["COMPONENT_SERIAL_NBR", "component_serial_nbr"],
    "desc": ["COMPONENT_DESCRIPTION", "component_description", "COMPONENT_TYPE_NAME"],
    "age":  ["component_age_days", "COMPONENT_AGE_DAYS"],
    "chg":  ["REPORTED_CHANGED_DTM", "reported_changed_dtm"],
})
print("resolved hw_config_current ->", {k: v for k, v in Rhw.items()})
age_col = (F.col(Rhw["age"]).cast("double") if Rhw["age"]
           else F.datediff(F.lit(RUN_DATE), F.to_date(F.col(Rhw["chg"]))).cast("double"))
serial = (hw_cfg.select(
              F.col(Rhw["dev"]).cast("string").alias("DEVICE_ID"),
              F.upper(F.col(Rhw["cat"]).cast("string")).alias("mars_device_category"),
              F.col(Rhw["ser"]).cast("string").alias("COMPONENT_SERIAL_NBR"),
              F.col(Rhw["desc"]).cast("string").alias("COMPONENT_TYPE_NAME"),
              F.greatest(age_col, F.lit(0.0)).alias("component_age_days"))
            .join(summary.select("DEVICE_ID", "total_hardware_oos_events", "total_chargeable_events", "chargeable_pct"),
                  "DEVICE_ID", "left")
            .join(comp_oos, ["DEVICE_ID", "COMPONENT_SERIAL_NBR"], "left")
            .fillna({"total_hardware_oos_events": 0, "total_chargeable_events": 0,
                     "chargeable_pct": 0.0, "component_oos_events": 0})
            .persist())
print("serial rows:", serial.count())
serial.show(6, truncate=False)


In [ ]:
# ── Cell 8 — Write compact outputs to LOCAL scratch (NOT silver/gold) ───────
summary_pd = summary.toPandas()
serial_pd  = serial.toPandas()
epi_pd     = epi_dev.toPandas()

summary_pd.to_parquet(f"{OUT_LOCAL}/device_oos_summary.parquet", index=False)
serial_pd.to_parquet(f"{OUT_LOCAL}/device_serial_oos.parquet", index=False)
epi_pd.to_parquet(f"{OUT_LOCAL}/device_oos_episodes.parquet", index=False)
summary_pd.to_csv(f"{OUT_LOCAL}/device_oos_summary.csv", index=False)
serial_pd.to_csv(f"{OUT_LOCAL}/device_serial_oos.csv", index=False)
print("wrote:", OUT_LOCAL, f"({len(summary_pd)} devices, {len(serial_pd)} serials, {len(epi_pd)} episodes)")


In [ ]:
# ── Cell 9 — Wire into the PS5 engine (in-memory override; NO table writes) ──
# The engine's load_table() honours a globals()["_SYNTH"] dict: any table there is served
# from the DataFrame instead of S3. We register ONLY the failure spine + component table,
# so every OTHER source still loads from silver normally.
import numpy as np, pandas as pd
assert "CONFIG" in globals(), "Run the PS5 engine cell first (CONFIG must exist)."

oos_engine = epi_pd.copy()
oos_engine["device_category"]    = oos_engine["mars_device_category"]
oos_engine["fault_state"]        = "Set"
oos_engine["is_oos_event"]       = True
oos_engine["is_device_fault"]    = True
oos_engine["is_commanded_oos"]   = False
oos_engine["is_maintenance_oos"] = False
serial_engine = serial_pd.copy()

globals().setdefault("_SYNTH", {})
globals()["_SYNTH"][CONFIG["FAILURE_TABLE"]]   = oos_engine       # replaces silver.device_failures (OOS, all types, min-outage)
globals()["_SYNTH"][CONFIG["COMPONENT_TABLE"]] = serial_engine    # replaces gold.device_ps5_component (true per-serial)
CONFIG["EVENT_DEF_VERSION"] = f"2026-07-24.oos_spine.minoutage{int(MIN_OUTAGE_MIN)}"

print("Wired _SYNTH override:")
print("  FAILURE_TABLE  ->", CONFIG["FAILURE_TABLE"],   "rows:", len(oos_engine))
print("  COMPONENT_TABLE->", CONFIG["COMPONENT_TABLE"], "rows:", len(serial_engine))
print("Now re-run:  results = main(synthetic=False)")


## How to run, and what changes

1. Run **Cell 1–8** (Spark). Check three lines: **`outage source`** (which method computed the duration), **`filters applied`** (real filters, not "NONE APPLIED"), and the **episodes_raw → episodes_kept** table — for validators you should see a big drop (the sub-5-min blips), and `chargeable_pct` ≈ 6–8% for TVM/gates, 0 for validators.
2. Run **Cell 9** in the **same kernel as your PS5 engine**, then re-run `results = main(synthetic=False)`.

**What the min-outage filter fixes.** Validators were scoring a ~1-day RUL because every transient comms blip counted as an OOS "failure." Dropping OOS shorter than `MIN_OUTAGE_MIN` (Cell 2, default 5 min) leaves only outages that actually needed attention, so the Weibull median stops collapsing to ~1 day. Tune `MIN_OUTAGE_MIN` up (e.g. 10–15) if validators are still too dense, or to 0 to disable.

**Honest notes.**
- The duration is derived **adaptively**: a duration column if `device_event_enriched` has one, else a clear-timestamp column, else Set→Clear pairing (each Set to its next Clear per device+component). The `outage source` print tells you which path ran — if it says `UNAVAILABLE`, the filter was skipped and you should send me the real clear-time/duration column (or `device_mttr`'s join keys) and I'll wire it exactly.
- Per-serial is **counts + age + risk**, not a survival curve (component-level survival is 100% censored per the catalog). `component_oos_events` is the true per-component OOS count.
- **No silver/gold tables are touched** — outputs live in `PS5_oos_spine_outputs/` and the in-memory `_SYNTH` override.
